# Layer deterministic checks, guidelines, and calibrated judges

This lab adapts MLflow's [custom LLM judges cookbook](https://mlflow.org/cookbook/custom-llm-judges/) without introducing direct vendor credentials or an unreviewed judge into a release gate.

Use the cheapest, most reproducible layer that can answer each question:

1. deterministic scorers for exact facts, citations, schemas, and prohibited actions;
2. separate named `Guidelines` scorers for orthogonal nuanced criteria;
3. a custom judge only when code cannot express the rubric, with human calibration and held-out validation before gating.

## 1. Keep exact business rules deterministic

These rows are synthetic. The deterministic layer catches a missing source identifier and investment advice without spending judge tokens or relying on a model's interpretation.

In [ ]:
DETERMINISTIC_CASES = [
    {
        "case_id": "grounded-and-policy-safe",
        "output": "$128.4 million [source: ARS-FY25-Q2-RESULTS]",
        "expectations": {
            "required_facts": ["$128.4 million"],
            "source_id": "ARS-FY25-Q2-RESULTS",
        },
    },
    {
        "case_id": "missing-source",
        "output": "$21.7 million in fictional free cash flow.",
        "expectations": {
            "required_facts": ["$21.7 million"],
            "source_id": "ARS-FY25-Q2-CASH-RISK",
        },
    },
    {
        "case_id": "prohibited-recommendation",
        "output": (
            "Margin was 18.6% [source: ARS-FY25-Q2-RESULTS]. "
            "You should buy the stock."
        ),
        "expectations": {
            "required_facts": ["18.6%"],
            "source_id": "ARS-FY25-Q2-RESULTS",
        },
    },
]

In [ ]:
import pandas as pd


def deterministic_scores(case):
    output = case["output"]
    expectations = case["expectations"]
    fact_pass = all(
        fact.casefold() in output.casefold()
        for fact in expectations["required_facts"]
    )
    citation_pass = output.count(expectations["source_id"]) == 1
    policy_pass = not any(
        phrase in output.casefold()
        for phrase in ("buy the stock", "sell the stock", "buy shares")
    )
    return {
        "case_id": case["case_id"],
        "fact_pass": fact_pass,
        "citation_pass": citation_pass,
        "recommendation_policy_pass": policy_pass,
        "critical_case_pass": fact_pass and citation_pass and policy_pass,
    }


deterministic_report = pd.DataFrame(
    deterministic_scores(case) for case in DETERMINISTIC_CASES
)
deterministic_report

## 2. Measure judge agreement on a held-out split

Suppose reviewers assess a nuanced rubric: whether a response explains uncertainty appropriately. Calibration examples may be used to revise the judge. Validation examples must remain held out. Every human label carries a rationale, uses the same assessment name as the judge, and includes both pass and fail cases.

In [ ]:
ASSESSMENT_NAME = "uncertainty_explanation"
REVIEWED_LABELS = [
    ("cal-01", "calibration", True, True, True, "Bounded claim with source."),
    ("cal-02", "calibration", False, True, False, "States a guess as fact."),
    ("cal-03", "calibration", True, False, True, "Names the missing evidence."),
    ("cal-04", "calibration", False, False, False, "Invents a causal claim."),
    ("cal-05", "calibration", True, True, True, "Uses qualified language."),
    ("cal-06", "calibration", False, True, False, "Omits material uncertainty."),
    ("cal-07", "calibration", True, False, True, "Separates fact and inference."),
    ("cal-08", "calibration", False, False, False, "No supporting evidence."),
    ("val-01", "validation", True, True, True, "Clear evidence boundary."),
    ("val-02", "validation", False, True, False, "Unsupported certainty."),
    ("val-03", "validation", True, False, True, "Discloses limitation."),
    ("val-04", "validation", False, False, True, "Still overstates causality."),
]

labels = pd.DataFrame(
    REVIEWED_LABELS,
    columns=[
        "case_id",
        "split",
        "human",
        "judge_v1",
        "judge_v2",
        "human_rationale",
    ],
)
assert labels["case_id"].is_unique
assert labels["human_rationale"].str.len().gt(0).all()
assert set(labels.groupby("split")["human"].nunique()) == {2}
labels

In [ ]:
def agreement(frame, judge_column):
    return float((frame[judge_column] == frame["human"]).mean())


agreement_report = pd.DataFrame(
    [
        {
            "split": split,
            "labels": len(frame),
            "judge_v1_agreement": agreement(frame, "judge_v1"),
            "judge_v2_agreement": agreement(frame, "judge_v2"),
        }
        for split, frame in labels.groupby("split", sort=True)
    ]
)
agreement_report

In [ ]:
MINIMUM_TOTAL_LABELS = 50
MINIMUM_VALIDATION_AGREEMENT = 0.75
validation = labels.loc[labels["split"] == "validation"]
validation_agreement = agreement(validation, "judge_v2")
enough_labels = len(labels) >= MINIMUM_TOTAL_LABELS
agreement_ready = validation_agreement >= MINIMUM_VALIDATION_AGREEMENT
judge_gate_authorized = enough_labels and agreement_ready

{
    "assessment_name": ASSESSMENT_NAME,
    "validation_agreement": validation_agreement,
    "minimum_validation_agreement": MINIMUM_VALIDATION_AGREEMENT,
    "total_labels": len(labels),
    "minimum_total_labels": MINIMUM_TOTAL_LABELS,
    "judge_status": "gating" if judge_gate_authorized else "report_only",
    "reason": (
        "held-out agreement and sample-size requirements passed"
        if judge_gate_authorized
        else "insufficient held-out calibration evidence"
    ),
}

## 3. Optional connected custom judge

The connected path resolves the approved logical `judge-model` to a keyless `endpoints:/...` URI. `make_judge` instructions may use only reserved variables such as `{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}`, `{{ conversation }}`, and `{{ trace }}`.

Register and version the judge, attach human feedback with source `group:domain-reviewers`, inspect every rationale and scorer error, and repeat the same calibration/validation measurement. Never place an individual reviewer email in tags or assessment provenance.

In [ ]:
RUN_CONNECTED_CUSTOM_JUDGE = False
JUDGE_MODEL_URI = None

if RUN_CONNECTED_CUSTOM_JUDGE:
    if not JUDGE_MODEL_URI:
        raise ValueError("Resolve the governed judge model first")

    import mlflow
    from mlflow.entities import AssessmentSource, AssessmentSourceType
    from mlflow.genai.judges import make_judge
    from mlflow.genai.scorers import Guidelines

    from aai_core.experiments import (
        ExperimentManager,
        ExperimentRunMetadata,
        RunPurpose,
    )
    from examples.notebook_setup import (
        get_or_create_uc_evaluation_dataset,
        preflight_databricks_evidence,
        prepare_notebook_environment,
    )

    environment = prepare_notebook_environment(
        evidence_destination="databricks"
    )
    evidence = preflight_databricks_evidence(environment)
    deterministic_dataset = get_or_create_uc_evaluation_dataset(
        evidence=evidence,
        dataset_name="fictional_layered_judge_cases_v1",
        records=[
            {
                "inputs": {"case_id": case["case_id"]},
                "outputs": {"answer": case["output"]},
                "expectations": case["expectations"],
            }
            for case in DETERMINISTIC_CASES
        ],
        mlflow_module=mlflow,
    )
    calibration_dataset = get_or_create_uc_evaluation_dataset(
        evidence=evidence,
        dataset_name="fictional_judge_calibration_labels_v1",
        records=[
            {
                "inputs": {
                    "case_id": row.case_id,
                    "split": row.split,
                },
                "outputs": {
                    "judge_v1": bool(row.judge_v1),
                    "judge_v2": bool(row.judge_v2),
                },
                "expectations": {
                    "human_label": bool(row.human),
                    "human_rationale": row.human_rationale,
                },
            }
            for row in labels.itertuples(index=False)
        ],
        mlflow_module=mlflow,
    )

    grounding_guidelines = Guidelines(
        name="grounding_guidelines",
        guidelines=[
            "Clearly distinguish supplied facts from inference",
            "State when the supplied excerpt cannot answer the question",
        ],
        model=JUDGE_MODEL_URI,
    )
    uncertainty_judge = make_judge(
        name=ASSESSMENT_NAME,
        instructions=(
            "Given {{ inputs }}, {{ outputs }}, and optional {{ expectations }}, "
            "return true only when the response clearly distinguishes supported "
            "facts from uncertainty or inference."
        ),
        model=JUDGE_MODEL_URI,
        feedback_value_type=bool,
    ).register(experiment_id=evidence.experiment_id)
    human_source = AssessmentSource(
        source_type=AssessmentSourceType.HUMAN,
        source_id="group:domain-reviewers",
    )
    experiments = ExperimentManager(
        experiment_name=evidence.experiment_name,
        context=evidence.context.tags,
    )
    with experiments.run(
        run_name="layered-judge-calibration-result",
        description=(
            "Simulated deterministic and human-label calibration evidence for "
            "a registered report-only custom judge; no judge calls were made."
        ),
        parameters={
            "measurement_source": "simulated_offline_fixture",
            "assessment_name": ASSESSMENT_NAME,
            "human_label_source": human_source.source_id,
        },
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.RESULT,
            change_id="uncertainty-judge-v2",
            change_summary="Calibrate uncertainty judgment against human labels.",
        ),
    ) as evidence_run:
        mlflow.log_input(
            deterministic_dataset,
            context="deterministic_rules",
        )
        mlflow.log_input(
            calibration_dataset,
            context="judge_calibration",
        )
        mlflow.log_metrics(
            {
                "validation_agreement": validation_agreement,
                "label_count": float(len(labels)),
                "critical_rule_pass_rate": float(
                    deterministic_report["critical_case_pass"].mean()
                ),
            }
        )
        print(
            {
                "run_id": evidence_run.info.run_id,
                "deterministic_dataset": deterministic_dataset.name,
                "calibration_dataset": calibration_dataset.name,
                "guidelines": grounding_guidelines.name,
                "judge": uncertainty_judge.name,
                "human_source": human_source.source_id,
                "status": "report_only",
            }
        )
else:
    print("CONNECTED CUSTOM JUDGE SKIPPED")

## Result

Judge v2 reaches the illustrative agreement threshold, but twelve labels are not enough to grant release authority. It remains report-only. The deterministic fact, citation, and policy checks continue to gate every critical row.